<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 2 章：练习解决方案

此笔记本中使用的库：

In [1]:
%pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt  # 安装项目所需的所有依赖

from importlib.metadata import version  # 从标准库导入用于查询已安装包版本的函数
used_libraries = [  # 列出要检查版本的库
    "reasoning_from_scratch",  # 本书/项目的核心库
    "torch",  # PyTorch 深度学习框架
    "tokenizers"  # Hugging Face 的分词库
]

for lib in used_libraries:  # 遍历每个库名
    print(f"{lib} version: {version(lib)}")  # 查询并打印对应库的已安装版本

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 146.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 114.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.3 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0


reasoning_from_scratch version: 0.1.10
torch version: 2.9.0+cu126
tokenizers version: 0.22.1


&nbsp;
## 练习 2.1：对未知单词进行编码

In [2]:
#下载 Qwen3 小模型的基础版 tokenizer 文件，保存到 qwen3 目录  # 下载 Qwen3 基础版的分词器
#然后将一串文本解码成token ID 列表                     # 编码文本为 token ID 列表
#最后解码并打印成 "[id] --> 对应文本片段" 的形式         # 再逐个解码并打印映射关系
from pathlib import Path  # 导入 Path 处理跨平台路径

from reasoning_from_scratch.qwen3 import (  # 从 qwen3 模块导入下载函数和分词器类
    download_qwen3_small,
    Qwen3Tokenizer,
)

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")  # 下载基础版分词器文件到 qwen3 目录

tokenizer_path = Path("qwen3") / "tokenizer-base.json"  # 构建分词器文件路径
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)  # 加载分词器实例

prompt = "Hello, Ardwarklethyrx. Haus und Garten."  # 要编码的输入文本
input_token_ids_list = tokenizer.encode(prompt)  # 将文本编码为 token ID 列表

for i in input_token_ids_list:  # 遍历每个 token ID
    print(f"{[i]} --> {tokenizer.decode([i])}")  # 打印 ID 及其解码得到的文本片段


[9707] --> Hello
[11] --> ,
[1644] -->  Ar
[29406] --> dw
[838] --> ark
[273] --> le
[339] --> th
[10920] --> yr
[87] --> x
[13] --> .
[47375] -->  Haus
[2030] -->  und
[93912] -->  Garten
[13] --> .


- 未知单词会被拆分成更小的子词甚至单个 token，这样 tokenizer 和 LLM 就能处理任何输入。
- 德语单词（Haus und Garten）在这里没有被拆分，说明 tokenizer 训练时见过德语文本，模型本身也很可能包含德语训练数据。

&nbsp;
## 练习 2.2：流式生成 Tokens

In [4]:
#下载并加载 Qwen3 0.6B 基座模型与分词器到计算设备，然后把输入提示编码成张量以便模型推理  # 说明本段目的：下载/加载模型与分词器并准备输入
from pathlib import Path  # 跨平台路径处理
import torch  # 深度学习框架

from reasoning_from_scratch.ch02 import (  # 从 ch02 工具模块导入设备选择函数
    get_device
)
from reasoning_from_scratch.qwen3 import (  # 导入下载函数、分词器、模型类及 0.6B 配置
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()  # 自动选择可用的计算设备（GPU/MPS/XPU/CPU）

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")  # 下载基础版 Qwen3 0.6B 模型和分词器到 qwen3 目录

tokenizer_path = Path("qwen3") / "tokenizer-base.json"  # 分词器文件路径
model_path = Path("qwen3") / "qwen3-0.6B-base.pth"  # 模型权重文件路径

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)  # 加载分词器
model = Qwen3Model(QWEN_CONFIG_06_B)  # 按 0.6B 配置初始化模型
model.load_state_dict(torch.load(model_path))  # 加载预训练权重

model.to(device)  # 将模型移动到所选设备

prompt = "Explain large language models in a single sentence."  # 要编码的提示文本
input_token_ids_tensor = torch.tensor(  # 将编码后的 token ID 列表转为张量
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)  # 增加 batch 维，形状变为 (1, seq_len)


Using NVIDIA CUDA GPU
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


- 下面更新后的 `generate_text_basic_stream` 函数几乎与第 2 章中的`generate_text_basic` 函数完全相同；差异之处已通过注释标出。

In [5]:
#函数以流式方式token生成文本：逐步推理模型输出、在遇到结束符或达到上限前逐个追加并立即解码打印。
@torch.inference_mode()  # 关闭梯度计算，加速推理
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None
):
    # input_length = token_ids.shape[1]  # 原序列长度（此处未使用）
    model.eval()  # 切换评估模式

    for _ in range(max_new_tokens):  # 逐个生成新 token，最多生成指定数量
        out = model(token_ids)[:, -1]  # 前向推理并取当前序列末尾位置的 logits
        next_token = torch.argmax(out, dim=-1, keepdim=True)  # 选取概率最高的下一个 token

        if (eos_token_id is not None
                and next_token.item() == eos_token_id):  # 若生成 EOS，则提前结束
            break

        yield next_token  # 新功能：每生成一个 token 就立即返回它（流式）

        token_ids = torch.cat([token_ids, next_token], dim=1)  # 将生成的 token 追加到输入序列
    # 返回 token_ids[:, input_length:]  # 若需要返回新增部分，可取消注释

for token in generate_text_basic_stream(  # 调用流式生成函数
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=50,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()  # 去掉 batch 维，转为列表
    print(
        tokenizer.decode(token_id),  # 解码并打印当前生成的 token 文本
        end="",
        flush=True  # 立即输出，不缓存
    )


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

- 提供一个支持 KV 缓存的版本：

In [6]:
from reasoning_from_scratch.qwen3 import KVCache  # 导入用于存储注意力键值的缓存类

@torch.inference_mode()  # 关闭梯度计算，加速推理
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    # input_length = token_ids.shape[1]  # 原序列长度（此处未使用）
    model.eval()  # 切换为评估模式
    cache = KVCache(n_layers=model.cfg["n_layers"])  # 为每层初始化 KV 缓存
    model.reset_kv_cache()  # 重置模型内部 KV 缓存

    out = model(token_ids, cache=cache)[:, -1]  # 首次前向并取末尾位置 logits
    for _ in range(max_new_tokens):  # 逐个生成，最多生成指定数量
        next_token = torch.argmax(out, dim=-1, keepdim=True)  # 选取概率最高的下一个 token

        if (eos_token_id is not None
                and next_token.item() == eos_token_id):  # 若生成 EOS，则提前停止
            break

        yield next_token  # 新：生成一个 token 就立刻产出
        # token_ids = torch.cat([token_ids, next_token], dim=1)  # 流式场景下无需累积全序列
        out = model(next_token, cache=cache)[:, -1]  # 增量前向，继续生成

    # return token_ids[:, input_length:]  # 若需返回新增部分，可取消注释

for token in generate_text_basic_stream_cache(  # 调用带缓存的流式生成
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=50,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()  # 去掉 batch 维，转成列表
    print(
        tokenizer.decode(token_id),  # 解码当前生成的 token 文本
        end="",  # 紧凑输出
        flush=True  # 立即刷新
    )


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

&nbsp;
## Exercise 2.3：在 GPU 设备上运行代码

- 只需删除第 2.5 节中的 `device = torch.device("cpu")` 这一行，然后重新运行代码

- 为方便起见，下面提供了一个来自第 2 章相关代码的最小自包含示例

In [ ]:
#model.to(device) 把模型权重和缓冲区迁移到指定的计算设备（CPU 或 GPU），以便后续推理时与 device 保持一致。  # 说明本段目的
from pathlib import Path  # 处理跨平台文件路径
import torch  # 深度学习框架 PyTorch

from reasoning_from_scratch.ch02 import (  # 从 ch02 工具模块导入设备与生成相关函数
    get_device,
    generate_text_basic,
    generate_text_basic_cache,
    generate_stats
)
from reasoning_from_scratch.qwen3 import (  # 导入下载函数、分词器、模型类及 0.6B 配置
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()  # 自动选择可用的计算设备
device = torch.device("cpu")  # 强制使用 CPU（覆盖上面自动选择的结果）

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")  # 下载基础版模型和分词器到 qwen3 目录

tokenizer_path = Path("qwen3") / "tokenizer-base.json"  # 分词器文件路径
model_path = Path("qwen3") / "qwen3-0.6B-base.pth"  # 模型权重文件路径

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)  # 加载分词器
model = Qwen3Model(QWEN_CONFIG_06_B)  # 按 0.6B 配置初始化模型
model.load_state_dict(torch.load(model_path))  # 加载预训练权重

model.to(device);  # 将模型迁移到指定设备（此处为 CPU）


Using NVIDIA CUDA GPU
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [8]:
prompt = "Explain large language models in 1 sentence."  # 要编码的提示文本
input_token_ids_tensor = torch.tensor(  # 将提示编码为 token ID 张量并放到目标设备
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)  # 增加 batch 维，形状变为 (1, seq_len)


In [9]:
#计时一次 generate_text_basic 调用的生成过程，并在结束后输出生成的 token 数和统计信息。  # 说明：计时生成并输出统计
import time  # 导入标准库 time 用于计时

start_time = time.time()  # 记录开始时间

max_new_tokens = 100  # 限制最多生成 100 个新 token
output_token_ids_tensor = generate_text_basic(  # 调用贪心生成函数
    model=model,  # 使用已加载的模型
    token_ids=input_token_ids_tensor,  # 编码后的输入提示
    max_new_tokens=max_new_tokens,  # 最大生成数量
    eos_token_id=tokenizer.eos_token_id,  # 遇到 EOS 提前停止
)
end_time = time.time()  # 记录结束时间

print(f"Output length: {output_token_ids_tensor.numel()}")  # 打印生成的 token 总数
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 输出耗时、速率等统计


Output length: 41
Time: 21.11 sec
1 tokens/sec
Max CUDA memory allocated: 1.48 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


In [10]:
start_time = time.time()  # 记录开始时间

output_token_ids_tensor = generate_text_basic_cache(  # 使用带 KV 缓存的生成函数
    model=model,  # 已加载的模型
    token_ids=input_token_ids_tensor,  # 编码后的输入提示
    max_new_tokens=max_new_tokens,  # 最大生成 token 数
    eos_token_id=tokenizer.eos_token_id,  # 遇到 EOS 提前停止
)
end_time = time.time()  # 记录结束时间

generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 输出耗时、速率和显存等统计


Time: 3.06 sec
13 tokens/sec
Max CUDA memory allocated: 0.01 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


In [11]:
if device.type == "mps":
    print(f"`torch.compile` is not supported for the {model.__class__.__name__} model on MPS (Apple Silicon) as of this writing.")
    model_compiled = model
    # 进行分配，以便如果有人使用“运行所有单元格”，笔记本不会在此停止
else:
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 8):
# 这可以避免重新触发模型重新编译
# 在 PyTorch 2.8 及更高版本中
# 如果模型包含类似 self.pos = self.pos + 1 的代码
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        
    model_compiled = torch.compile(model)

In [12]:
#该循环先运行一次热身推理，再进行两次计时推理  # 说明：总共跑 3 次，首轮热身
#每次调用编译后的 generate_text_basic 计算生成结果并用 generate_stats 打印性能统计  # 说明：使用编译模型并输出统计
for i in range(3):  # 迭代 0,1,2 三次
    start_time = time.time()  # 记录开始时间
    output_token_ids_tensor = generate_text_basic(  # 调用贪心生成
        model=model_compiled,  # 使用 torch.compile 过的模型
        token_ids=input_token_ids_tensor,  # 输入提示的 token 张量
        max_new_tokens=max_new_tokens,  # 生成的最大 token 数
        eos_token_id=tokenizer.eos_token_id  # 遇到 EOS 提前停止
    )
    end_time = time.time()  # 记录结束时间

    if i == 0:
        print("Warm-up run")  # 第一次为热身，不计性能
    else:
        print(f"Timed run {i}:")  # 标记第几次计时运行
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 打印耗时、速率等统计

    print(f"\n{30*'-'}\n")  # 分隔线，便于阅读输出


Warm-up run
Time: 154.90 sec
0 tokens/sec
Max CUDA memory allocated: 0.01 GB

 Large language models are artificial intelligence systems that use vast amounts of text data to understand and generate human-like language, enabling them to perform tasks such as translation, summarization, and question answering.

------------------------------

Timed run 1:
Time: 17.06 sec
2 tokens/sec
Max CUDA memory allocated: 0.01 GB

 Large language models are artificial intelligence systems that use vast amounts of text data to understand and generate human-like language, enabling them to perform tasks such as translation, summarization, and question answering.

------------------------------

Timed run 2:
Time: 17.08 sec
2 tokens/sec
Max CUDA memory allocated: 0.01 GB

 Large language models are artificial intelligence systems that use vast amounts of text data to understand and generate human-like language, enabling them to perform tasks such as translation, summarization, and question answering.



In [13]:
for i in range(3):  # 总共运行 3 次：第一次热身，后两次计时
    start_time = time.time()  # 记录开始时间
    output_token_ids_tensor = generate_text_basic_cache(  # 使用带 KV 缓存的生成函数
        model=model_compiled,  # 已编译模型
        token_ids=input_token_ids_tensor,  # 输入提示的 token 张量
        max_new_tokens=max_new_tokens,  # 最大生成 token 数
        eos_token_id=tokenizer.eos_token_id  # 遇到 EOS 提前结束
    )
    end_time = time.time()  # 记录结束时间

    if i == 0:
        print("Warm-up run")  # 第一次为热身，不计性能
    else:
        print(f"Timed run {i}:")  # 标注当前计时轮次
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 输出耗时、速率、显存等统计

    print(f"\n{30*'-'}\n")  # 分隔线，便于阅读


Warm-up run
Time: 201.36 sec
0 tokens/sec
Max CUDA memory allocated: 0.01 GB

 Large language models are artificial intelligence systems that use vast amounts of text data to understand and generate human-like language, enabling them to perform tasks such as translation, summarization, and question answering.

------------------------------

Timed run 1:
Time: 1.70 sec
22 tokens/sec
Max CUDA memory allocated: 0.01 GB

 Large language models are artificial intelligence systems that use vast amounts of text data to understand and generate human-like language, enabling them to perform tasks such as translation, summarization, and question answering.

------------------------------

Timed run 2:
Time: 1.62 sec
23 tokens/sec
Max CUDA memory allocated: 0.01 GB

 Large language models are artificial intelligence systems that use vast amounts of text data to understand and generate human-like language, enabling them to perform tasks such as translation, summarization, and question answering.



| Tokens Generated  | Mode              | Hardware        | Tokens/sec    | GPU Memory (VRAM) |
|-------------------|-------------------|-----------------|---------------|-------------------|
| 41                | Regular           | Mac Mini M4 CPU | 6             | -                 |
| 41                | Regular compiled  | Mac Mini M4 CPU | 6             | -                 |
| 41                | KV cache          | Mac Mini M4 CPU | 28            | -                 |
| 41                | KV cache compiled | Mac Mini M4 CPU | 68            | -                 |
|                   |                   |                 |               |                   |
| 41                | Regular           | Mac Mini M4 GPU | 17            | -                 |
| 41                | Regular compiled  | Mac Mini M4 GPU | InductorError | -                 |
| 41                | KV cache          | Mac Mini M4 GPU | 18            | -                 |
| 41                | KV cache compiled | Mac Mini M4 GPU | InductorError | -                 |
|                   |                   |                 |               |                   |
| 41                | Regular           | NVIDIA H100 GPU | 51            | 1.55 GB           |
| 41                | Regular compiled  | NVIDIA H100 GPU | 164           | 1.81 GB           |
| 41                | KV cache          | NVIDIA H100 GPU | 48            | 1.52 GB           |
| 41                | KV cache compiled | NVIDIA H100 GPU | 141           | 1.81 GB           |